# Streaming ML Demo

Train a pipeline **one chunk at a time** on wine quality data.

This notebook loads data, splits it into chunks, and calls `partial_fit` in a loop.
Right now it uses a dummy classifier — swap in `DecisionTreeClassifier` / `RandomForestClassifier` later.

## Setup

Run the cell below first. It finds the package folder automatically (works when the notebook lives in `demo/`).

Alternatively install once from the `NumCompute` folder:

```python
# !pip install -e ..
```

In [ ]:
import sys
from pathlib import Path

import numpy as np


def _find_package_root():
    # notebook is usually run from NumCompute/demo/
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent / "NumCompute",
    ]
    for path in candidates:
        if (path / "numcompute").is_dir() and (path / "numcompute_stream").is_dir():
            return path.resolve()
    raise ImportError(
        "Could not find numcompute. cd to NumCompute/demo or run: pip install -e .."
    )


root = _find_package_root()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from numcompute.io import load_csv
from numcompute_stream.metrics import StreamingAccuracy
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.preprocessing import StandardScaler

np.set_printoptions(linewidth=120, precision=4, suppress=True)
print("using package root:", root)

## 1. Load data

White wine dataset: 11 features + quality score (last column).
We binarise quality: **1** if quality >= 6, else **0**.

In [ ]:
wine = load_csv("../assets/winequality-white.csv", dtype=np.float32, delimiter=";")
print("raw shape:", wine.shape)

X = wine[:, :11]
y = (wine[:, 11] >= 6).astype(int)

print("X shape:", X.shape)
print("class balance (0, 1):", np.bincount(y))

## 2. Split into chunks

Simulates streaming data arriving in batches.

In [ ]:
chunk_size = 50
chunks = [
    (X[i:i + chunk_size], y[i:i + chunk_size])
    for i in range(0, len(X), chunk_size)
]

print(f"{len(chunks)} chunks of up to {chunk_size} rows")
print("first chunk shapes:", chunks[0][0].shape, chunks[0][1].shape)

## 3. Stand-in model

Replace this with `DecisionTreeClassifier` once Phase 3 is done.

In [ ]:
class DummyClassifier:
    """Predicts the majority class seen so far"""

    def __init__(self):
        self.classes_ = None
        self.majority_ = None

    def partial_fit(self, X, y):
        self.classes_ = np.unique(y)
        self.majority_ = np.bincount(y.astype(int)).argmax()
        return self

    def predict(self, X):
        return np.full(len(X), self.majority_)

    def fit(self, X, y=None):
        return self.partial_fit(X, y)

## 4. Train incrementally

Each chunk: scale -> partial_fit model -> score on that chunk.

In [ ]:
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", DummyClassifier()),
])

chunk_accuracies = []
cumulative_metric = StreamingAccuracy()

for i, (X_chunk, y_chunk) in enumerate(chunks):
    pipe.partial_fit(X_chunk, y_chunk)
    y_pred = pipe.predict(X_chunk)

    chunk_acc = float(np.mean(y_pred == y_chunk))
    chunk_accuracies.append(chunk_acc)
    cumulative_metric.update(y_chunk, y_pred)

    print(
        f"chunk {i + 1:2d}/{len(chunks)} | "
        f"chunk acc={chunk_acc:.3f} | "
        f"cumulative acc={cumulative_metric.result():.3f} | "
        f"majority={pipe.named_steps['model'].majority_}"
    )

## 5. Summary

In [ ]:
print(f"chunks trained: {len(chunks)}")
print(f"final chunk accuracy: {chunk_accuracies[-1]:.3f}")
print(f"final cumulative accuracy: {cumulative_metric.result():.3f}")
print(f"scaler mean (first 3 features): {pipe.named_steps['scale'].mean_[:3]}")

## Next steps (reuse this notebook later)

- Swap `DummyClassifier` -> `DecisionTreeClassifier` / `RandomForestClassifier`
- Add second pipeline and compare tree vs forest
- Replace print loop with `StreamTrainer` + `visualise.plot_metric_over_time` / `compare_models`